In [6]:
import os
import sys

sys.path.append(os.path.join(os.getcwd(), "preprocessing", "label_mapping"))
from label_mapping import build_labeled_dataset

# mapping the label
df = build_labeled_dataset()


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26179 entries, 0 to 26178
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_path  26179 non-null  object
 1   label_it    26179 non-null  object
 2   label_en    26179 non-null  object
dtypes: object(3)
memory usage: 613.7+ KB


In [8]:
sys.path.append(os.path.join(os.getcwd(), "preprocessing", "data_loader"))
from data_loader import build_train_val_test_generators

# ImageDataGenerator resizes each image on load and rescales pixels to [0, 1]
# (normalization). 128x128 instead of 224 to train much faster on CPU, at a
# small accuracy cost. 70/15/15 train/val/test split.
train_generator, val_generator, test_generator = build_train_val_test_generators(
    df, project_root=os.getcwd(), image_size=(128, 128)
)

print(
    f"Train batches: {len(train_generator)}, "
    f"Val batches: {len(val_generator)}, "
    f"Test batches: {len(test_generator)}"
)
train_generator.class_indices

Found 18325 validated image filenames belonging to 10 classes.
Found 3927 validated image filenames belonging to 10 classes.
Found 3927 validated image filenames belonging to 10 classes.
Train batches: 573, Val batches: 123, Test batches: 123


{'butterfly': 0,
 'cat': 1,
 'chicken': 2,
 'cow': 3,
 'dog': 4,
 'elephant': 5,
 'horse': 6,
 'sheep': 7,
 'spider': 8,
 'squirrel': 9}

In [ ]:
sys.path.append(os.path.join(os.getcwd(), "model"))
from baseline_cnn import build_baseline_cnn

# input_shape and num_classes both read from the generator so the model always
# stays in sync with the data (image_shape is e.g. (128, 128, 3)).
model = build_baseline_cnn(
    input_shape=train_generator.image_shape,
    num_classes=len(train_generator.class_indices),
)
model.summary()

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop once val_loss stops improving for 3 epochs, and restore the weights
# from the best epoch so we don't keep an overfitted final state.
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stopping],
)

Epoch 1/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 82s 141ms/step - accuracy: 0.2984 - loss: 1.9923 - val_accuracy: 0.3382 - val_loss: 1.9283
Epoch 2/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 92s 160ms/step - accuracy: 0.4748 - loss: 1.5510 - val_accuracy: 0.5360 - val_loss: 1.3971
Epoch 3/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 89s 156ms/step - accuracy: 0.5729 - loss: 1.2729 - val_accuracy: 0.5915 - val_loss: 1.2383
Epoch 4/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 92s 161ms/step - accuracy: 0.6186 - loss: 1.1269 - val_accuracy: 0.6101 - val_loss: 1.2011
Epoch 5/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 90s 157ms/step - accuracy: 0.6605 - loss: 1.0205 - val_accuracy: 0.6290 - val_loss: 1.1353
Epoch 6/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 93s 163ms/step - accuracy: 0.6818 - loss: 0.9290 - val_accuracy: 0.6481 - val_loss: 1.0686
Epoch 7/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 96s 167ms/step - accuracy: 0.7145 - loss: 0.8392 - val_accuracy: 0.6567 - val_loss: 1.0406
Epoch 8/30
573/573 ━━━━━━━━━━━━━━━━━━━━ 95s 166ms/step - accuracy: 0.7386 - loss: 0